# Meta-Learning with MOORE on MetaWorldThis notebook demonstrates the implementation of Mixture of Orthogonal Experts (MOORE) on the MetaWorld benchmark.The goal is to learn a policy that can quickly adapt to a variety of robotic manipulation tasks.

## 1. Setup and Imports

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npimport metaworldimport matplotlib.pyplot as pltfrom functools import partialimport randomimport syssys.path.append('..')from meta_rl.algorithms.moore import MOOREfrom meta_rl.utils.data_utils import collect_trajectories, process_trajectoriestry:    jax.devices('gpu')except RuntimeError:    jax.config.update('jax_platform_name', 'cpu')

## 2. Creating the MetaWorld Environment

In [ ]:
ml1 = metaworld.ML1('reach-v2')env = ml1.train_classes['reach-v2']()task = random.choice(ml1.train_tasks)env.set_task(task)obs_dim = env.observation_space.shape[0]action_dim = env.action_space.shape[0]

## 3. MOORE Training Loop

In [ ]:
META_ITERATIONS = 5META_BATCH_SIZE = 4TRAJECTORIES_PER_TASK = 2MAX_STEPS_PER_TRAJECTORY = 50INNER_LR = 0.01META_LR = 0.001NUM_EXPERTS = 2moore = MOORE(    action_dim=action_dim,    obs_dim=obs_dim,    num_experts=NUM_EXPERTS,    inner_lr=INNER_LR,    meta_lr=META_LR)rng = jax.random.PRNGKey(0)rng, key = jax.random.split(rng)meta_params, opt_state = moore.init_params(key)policy_fn = moore.network.applylosses = []print('Starting MOORE training...')for meta_iter in range(META_ITERATIONS):    support_batches = []    query_batches = []    tasks = random.sample(ml1.train_tasks, META_BATCH_SIZE)    for task in tasks:        env.set_task(task)                rng, key = jax.random.split(rng)        support_trajectories = collect_trajectories(env, meta_params, policy_fn, TRAJECTORIES_PER_TASK, MAX_STEPS_PER_TRAJECTORY, key)        support_batch = process_trajectories(support_trajectories, policy_fn, meta_params)        support_batches.append(support_batch)                rng, key = jax.random.split(rng)        query_trajectories = collect_trajectories(env, meta_params, policy_fn, TRAJECTORIES_PER_TASK, MAX_STEPS_PER_TRAJECTORY, key)        query_batch = process_trajectories(query_trajectories, policy_fn, meta_params)        query_batches.append(query_batch)        stacked_support_batch = jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *support_batches)    stacked_query_batch = jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *query_batches)    stacked_meta_batch = (stacked_support_batch, stacked_query_batch)        meta_params, opt_state, loss = moore.outer_update(meta_params, opt_state, stacked_meta_batch)    losses.append(loss)        if (meta_iter + 1) % 1 == 0:        print(f'Meta-iteration {meta_iter + 1}/{META_ITERATIONS}, Loss: {loss:.4f}')print('Training finished.')

## 4. Visualizing Results

In [ ]:
plt.figure(figsize=(10, 5))plt.plot(losses)plt.title('MOORE Meta-Loss over Training')plt.xlabel('Meta-Iteration')plt.ylabel('Loss')plt.grid(True)plt.show()